# Полный BART A100: инференс в Colab на T4

1. Скачайте [a100_full_best180000_colab.zip](https://github.com/ArtyomStepanov/nlp-hw2/releases/download/models-2026-10-24/a100_full_best180000_colab.zip) из GitHub Release через браузер.
2. Выберите **Runtime → Change runtime type → T4 GPU**.
3. Загрузите ZIP в `/content/` через боковую панель **Files → Upload** и дождитесь окончания загрузки.
4. Выполните ячейки сверху вниз. В конце скачайте JSONL с 1919 ответами для бота.

Архив содержит исходные веса **checkpoint-180000** полного BART (244 391 936 параметров), BPE8000, конфигурацию, код инференса и официальные вопросы. Контрольные суммы ZIP, manifest и всех файлов проверяются до генерации. GitHub-токен в Colab не нужен.

Настройки: FP16 autocast, beam 4, length penalty 1, batch 128, вход/выход до 256 токенов. Обучение и выполнение сгенерированного кода не запускаются. Текущее время генерации измеряется отдельно от установки зависимостей, проверки файлов и загрузки весов.


In [ ]:
# Сохраняем установленную в Colab сборку PyTorch/CUDA.
%pip -q install "transformers==5.10.2" "tokenizers==0.22.2" "datasets==4.6.0" safetensors


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import json
import stat
import subprocess
import sys
import zipfile

ARCHIVE = Path('/content/a100_full_best180000_colab.zip')
EXPECTED_BYTES = 978377281
EXPECTED_SHA256 = '52dde24e1984813b3b9030f59123a3e9b48a53a6e1a278bafa3d81fa4aa3e1e7'
EXPECTED_MANIFEST_SHA256 = '4ac0daa90326956e69152eb681131bd5889af8ad5876a80591a532a1d2ee81ba'
EXTRACT_ROOT = Path('/content/nlp_hw2_a100_release')
BUNDLE_ROOT = EXTRACT_ROOT / 'a100_full_best180000_t4'


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


if not ARCHIVE.is_file() or ARCHIVE.stat().st_size != EXPECTED_BYTES:
    raise ValueError('Загрузите весь ZIP через Files в /content/ и повторите эту ячейку.')
if sha256_file(ARCHIVE) != EXPECTED_SHA256:
    raise ValueError('SHA256 архива не совпадает с опубликованной версией.')
if EXTRACT_ROOT.exists():
    raise FileExistsError(
        'Архив уже распаковывался. Для повторного инференса выполните следующую ячейку; '
        'для новой распаковки перезапустите runtime.'
    )
with zipfile.ZipFile(ARCHIVE) as archive:
    members = archive.infolist()
    names = [entry.filename for entry in members]
    if len(names) != len(set(names)):
        raise ValueError('В архиве есть повторяющиеся пути.')
    if sum(entry.file_size for entry in members) > 2 * 1024**3:
        raise ValueError('Неожиданный размер распакованных файлов.')
    for entry in members:
        name = PurePosixPath(entry.filename)
        mode = entry.external_attr >> 16
        if (name.is_absolute() or '..' in name.parts or '\\' in entry.filename
                or not name.parts or name.parts[0] != 'a100_full_best180000_t4'
                or stat.S_ISLNK(mode)):
            raise ValueError(f'Небезопасный путь в архиве: {entry.filename}')
    if archive.testzip() is not None:
        raise ValueError('CRC-проверка ZIP не прошла.')
    archive.extractall(EXTRACT_ROOT)
if sha256_file(BUNDLE_ROOT / 'manifest.json') != EXPECTED_MANIFEST_SHA256:
    raise ValueError('SHA256 manifest не совпадает.')
# Проверяются SHA/размер каждого файла, веса, конфигурация и порядок вопросов.
# Эта команда не загружает модель на GPU и не запускает генерацию.
subprocess.run([
    sys.executable, str(BUNDLE_ROOT / 'benchmark_full_t4.py'),
    '--bundle-root', str(BUNDLE_ROOT), '--verify-only',
], check=True, cwd=BUNDLE_ROOT)
print('READY: архив и все файлы проверены. Bundle:', BUNDLE_ROOT)


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

# Для каждого запуска создаётся новая папка вне неизменного bundle.
OUTPUT = Path('/content') / (
    'a100_full_inference_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    + '_' + uuid4().hex[:8]
)
subprocess.run([
    sys.executable, str(BUNDLE_ROOT / 'benchmark_full_t4.py'),
    '--bundle-root', str(BUNDLE_ROOT),
    '--output-dir', str(OUTPUT), '--batch-size', '128',
], check=True, cwd=BUNDLE_ROOT)
metrics = json.loads((OUTPUT / 'metrics.json').read_text())
print('Результаты:', OUTPUT)
print('Время генерации, с:', round(metrics['total_seconds'], 3))
print('Внешнее время вызова, с:', round(metrics['wall_generation_seconds'], 3))
print('В пределах 180 секунд:', metrics['within_180_seconds'])
print('Пик памяти GPU, ГиБ:', round(metrics['peak_allocated_gib'], 3))


In [ ]:
from google.colab import files

SUBMISSION = OUTPUT / 'a100_full_best180000_beam4_t4.jsonl'
complete = json.loads((OUTPUT / 'complete.json').read_text())
rows = [json.loads(line) for line in SUBMISSION.read_text(encoding='utf-8').splitlines()]
if complete['status'] != 'complete' or len(rows) != 1919:
    raise ValueError('Генерация не завершена или число ответов неверно.')
if any(set(row) != {'code'} or not isinstance(row['code'], str) for row in rows):
    raise ValueError('Неверный формат JSONL.')
if sha256_file(SUBMISSION) != complete['submission_sha256']:
    raise ValueError('SHA256 итогового JSONL не совпадает.')
print('Для бота:', SUBMISSION.name, '| 1919 строк | поле code | порядок сохранён')
print('SHA256:', complete['submission_sha256'])
if not complete['within_180_seconds']:
    print('Этот прогон превысил лимит 180 секунд; проверьте фактическое время выше.')
files.download(str(SUBMISSION))
# Дополнительно можно скачать метрики и полный receipt через панель Files:
print('Архив метрик и результатов:', OUTPUT / 't4_results.zip')
